# 🎙️ Whisper LoRA Fine-Tuning Notebook

## What is this notebook?
This notebook fine-tunes **OpenAI Whisper Large v2** — a state-of-the-art speech recognition model — on Urdu, Pashto, and Punjabi audio using **LoRA** (Low-Rank Adaptation).

**LoRA** lets us teach the model new languages by updating only a tiny fraction of its parameters (~7%), instead of retraining all 1.6 billion from scratch. This saves GPU memory, time, and cost.

---

## 📋 Pipeline at a Glance
| Step | What happens |
|------|--------------|
| 1 | Configure GPU & install libraries |
| 2 | Import libraries |
| 3 | Set training configuration (paths, batch size, epochs…) |
| 4 | Load the Whisper processor |
| 5 | Load the dataset (CSV → train / val / test splits) |
| 6 | Define the audio pre-processing function |
| 7 | Run feature extraction & tokenisation (converts audio → numbers) |
| 8 | Filter out bad/too-long samples |
| 9 | Load Whisper model and attach LoRA adapters |
| 10 | Build data collator and evaluation metrics |
| 11 | Configure the Trainer |
| 12 | 🚀 Train the model |
| 13 | 💾 Save adapter + merged model |

---

> ▶ **Run cells top-to-bottom, in order.** Each cell depends on the ones above it.


## Step 1 — GPU & Environment Setup

**What this does:** Selects GPU #0 as the active device and enables expandable memory segments — a PyTorch setting that prevents out-of-memory crashes during long training runs by allocating GPU memory in flexible chunks rather than fixed blocks.

> **Non-technical:** Think of the GPU as the engine. This cell makes sure the engine is pointed at the right fuel tank and the fuel lines are flexible before we start driving.

> ⚠️ **Run this cell first, before anything else.**



In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Step 1b — Install Required Libraries

**What this does:** Downloads and installs all Python packages needed for this pipeline.

| Library | Purpose |
|---------|--------|
| `torch` / `torchaudio` | Core deep-learning engine |
| `transformers` | Whisper model + Seq2Seq trainer |
| `peft` | LoRA adapter implementation |
| `datasets` | Fast dataset loading & processing |
| `librosa` / `soundfile` | Load and process audio files |
| `evaluate` / `jiwer` / `sacrebleu` | WER and BLEU scoring |
| `tensorboard` | Real-time training visualisation |

> **Non-technical:** Like gathering all your ingredients before cooking. May take a few minutes the very first time.

> 💡 `Requirement already satisfied` messages are normal — they mean the library is already present.


In [2]:
# ! pip uninstall torch torchaudio torchvision transformers -y #if needed 
! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# ! pip uninstall datasets[audio] accelerate evaluate   jiwer tensorboard sentencepiece librosa soundfile sacrebleu  -y 
! pip install transformers datasets[audio] accelerate evaluate peft  jiwer tensorboard sentencepiece librosa soundfile sacrebleu torchaudio torchvision
! pip install --upgrade transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cu124


## Step 2 — Import Libraries

**What this does:** Loads the installed packages into the current Python session so the rest of the notebook can use them.

> **Non-technical:** Installing (Step 1b) puts tools on the shelf. Importing (this step) puts them on your workbench ready to use.


In [3]:
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch
from datasets import load_from_disk
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

/home/orgpu/Downloads/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 — Training Configuration

**What this does:** Defines every important setting for the training run in one place.

> 🔧 **If you want to customise the training, this is the only cell you need to edit.**

| Setting | Current value | What it means |
|---------|--------------|---------------|
| `MODEL_NAME` | `openai/whisper-large-v2` | Base model to start from |
| `RUN_NAME` | `whisper-large-v2_v1_240526` | Folder name for saved checkpoints |
| `NUM_EPOCHS` | 5 | How many times the model sees the full training set |
| `BATCH_SIZE` | 32 | Audio clips processed together per GPU step |
| `GRAD_ACCUM_STEPS` | 8 | Steps before a weight update (effective batch = 32×8 = 256) |
| `LEARNING_RATE` | 1e-4 | Size of each learning step |
| `MAX_LABEL_LENGTH` | 448 tokens | Maximum transcript length Whisper can generate |
| Languages | Lang1, Lang2, Lang3 | Languages the model will specialise in, "Make sure the languages are supported by Whisper " |

> **Non-technical:** This is the recipe card. All ingredient amounts and cooking times are written here before anything goes into the oven.


In [ ]:
MODEL_NAME = "openai/whisper-large-v2"
TRAIN_DATASET_PATH = "./processed/train"
VAL_DATASET_PATH = "./processed/val"
TEST_DATASET_PATH = "./processed/test"
BASE_MODEL    = "openai/whisper-large-v2"
RUN_NAME      = "whisper-large-v2_v1_240526"   

LANG_MAPPING = {
     "Lang1":    "lang1",
     "Lang2":  "lang2",
     "Lang3": "lang3",

    }
TASK_MAPPING = {
    "Transcribe": "transcribe",
}

OUTPUT_DIR = f"whisper_finetuning/{RUN_NAME}"
MAX_LABEL_LENGTH = 448
MAX_INPUT_LENGTH = 3000           # Whisper default
BATCH_SIZE = 32                    
GRAD_ACCUM_STEPS = 8              
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4

LANGUAGE = None
TASK = "transcribe"


## Step 4 — Load Whisper Processor

**What this does:** Loads the Whisper *processor* — a paired tool that:
- **Feature extractor**: converts raw audio waveforms into Mel spectrograms (number grids the model reads)
- **Tokeniser**: converts text transcriptions into token IDs (numbers) and back again

> **Non-technical:** The processor is the translator between the human world (audio files, text) and the model's world (grids of numbers). Without it, the model cannot understand your data.

> 💡 A warning about `HF_TOKEN` is safe to ignore — the model is publicly available without authentication.


In [5]:
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK,
)
print("\nLoading datasets from disk...")



Loading datasets from disk...


# Load Dataset


## Step 5 — Load the Dataset

**What this does:** Reads the three CSV files that describe your audio data and creates train, validation, and test splits.

Each CSV row contains:
- **`audio_file_path`** — path to the `.wav` audio clip
- **`text`** — the correct written transcription
- **`source_lang`** — language spoken (Urdu / Pashto / Punjabi)
- **`task`** — transcription or translation

| Split | Role |
|-------|------|
| **Train** | Examples the model learns from |
| **Validation** | Checked after each epoch to monitor progress — never used for learning |
| **Test** | Final evaluation only — the real exam |

> **Non-technical:** Train = practice questions, Validation = mock test, Test = the real exam.

> 🔧 **Update the CSV paths in the cell below if your files are stored in a different location.**


In [ ]:
TRAIN_CSV="/home/orgpu/Downloads/In-House_training/train.csv"
VAL_CSV="/home/orgpu/Downloads/In-House_training/val.csv"
TEST_CSV="/home/orgpu/Downloads/In-House_training/test.csv"

from datasets import load_dataset, Value

raw = load_dataset("csv", data_files={
    "train": TRAIN_CSV,
    "val":   VAL_CSV,
    "test":  TEST_CSV,
}, sep="," )

# Keep audio_path as plain string (not Audio feature) — we load manually
for split in ["train", "val", "test"]:
    raw[split] = raw[split].cast_column("audio_file_path", Value("string"))

print(raw)

### 5a — Preview: One Row from Each Split

**What this does:** Prints the first row from train, val, and test so you can visually confirm the data loaded correctly.

> Check that `audio_file_path`, `text`, `source_lang`, and `task` all have sensible values. If `text` is empty or paths look wrong, fix the CSV paths above.


In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n--- {split} ---")
    print(raw[split][0])

### 5b — Reload Processor in Multilingual Mode

**What this does:** Reloads the Whisper processor without locking it to a single language. This is necessary because our dataset spans multiple languages — if the processor were fixed to one language, it could not correctly tokenise the others.


In [8]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL)
print("Processor loaded.")

Processor loaded.


## Step 6 — Define Audio Pre-processing Function

**What this does:** Defines `prepare_dataset()` — a function that will be applied to every audio clip in the dataset. For each clip it:

1. Loads the `.wav` file from disk
2. Resamples to **16,000 Hz** (Whisper's required sample rate)
3. Converts to **mono** (single audio channel)
4. Trims to a maximum of **30 seconds** (Whisper's context limit)
5. Converts the waveform to a **Mel spectrogram** (the model's visual representation of sound)
6. **Tokenises** the text transcription into numeric IDs
7. Tags each clip with the correct **language** and **task** for the Whisper decoder
8. If a file is missing or corrupted → marks it as `is_valid=False` and skips it (no crash)

> **Non-technical:** This is the factory worker's job description. The next cell runs the factory — calling this function thousands of times, once per audio file.

> ℹ️ **Running this cell only defines the function. No files are processed yet.**


In [ ]:
import librosa
import numpy as np

TARGET_SR = 16_000   # Whisper requires 16 kHz
MAX_DURATION_S = 30  # Whisper's context window is 30 s


def prepare_dataset(batch):
    """Map function: audio file → input_features + tokenised labels."""

    audio_path = "/home/orgpu/Downloads/In-House_training/" + batch["audio_file_path"]

    try:
        audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"[SKIP] Cannot load '{audio_path}': {e}")
        batch["is_valid"]       = False
        batch["input_features"] = None
        batch["labels"]         = None
        batch["forced_decoder_ids"] = None
        return batch

    # Trim silence beyond 30 s (Whisper ignores it anyway)
    max_samples = MAX_DURATION_S * TARGET_SR
    audio = audio[:max_samples]

    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=TARGET_SR
    ).input_features[0]

    lang_raw  = str(batch["source_lang"]).strip().lower()
    task_raw  = str(batch["task"]).strip().lower()
    # print(lang_raw, task_raw)
   
    lang_name = LANG_MAPPING.get(lang_raw, "english")   # default: english
    
    task_name = TASK_MAPPING.get(task_raw, "transcribe") # default Task: transcribe "You can add translation if your dataset contain the translation alongside transcription"

    batch["forced_decoder_ids"] = processor.tokenizer.get_decoder_prompt_ids(
        language=lang_name,
        task=task_name,
    )

    # ── 4. Tokenise transcription ────────────────────────────────────────────
    batch["labels"] = processor.tokenizer(
        batch["text"],
        truncation=True,
        # return_overflowing_tokens=True #if needed
    ).input_ids

    batch["is_valid"] = True
    return batch


print("prepare_dataset() defined.")

prepare_dataset() defined.


## Step 7 — Feature Extraction & Tokenisation (Resumable Sharded Processing)

**What this does:** Runs `prepare_dataset()` across all audio files in train, val, and test.

Processing uses **shards** (chunks of up to 50,000 rows) for two reasons:
- **Crash-safe**: completed shards are saved to disk immediately — if interrupted, only unfinished shards re-run
- **Memory-safe**: avoids loading the entire dataset into RAM at once

**Output messages explained:**
| Message | Meaning |
|---------|--------|
| `[RUN] Shard X/Y — rows A:B` | Currently processing rows A through B |
| `[SKIP] Shard X/Y already done` | Shard finished in a previous run — skipping |
| `Kept N/M rows after filtering` | M rows processed, N had valid audio |
| `✓ train done — N rows saved` | Split complete, written to `./Mapped Dataset/train` |

> **Non-technical:** Like processing a large photo library in batches — if the computer crashes at batch 5, you don't need to redo batches 1–4.

> ⏱️ **This cell can take a long time** (minutes to hours depending on dataset size). Progress bars will show status.


In [ ]:
import os
from datasets import load_from_disk, concatenate_datasets

SHARD_SIZE = 50_000  # tune down if RAM is tight (librosa is memory heavy)

def process_split_resumable(dataset_split, split_name, save_base="./Mapped Dataset"):
    shard_dir = f"{save_base}/{split_name}_shards"
    os.makedirs(shard_dir, exist_ok=True)
    
    total = len(dataset_split)
    num_shards = (total + SHARD_SIZE - 1) // SHARD_SIZE
    print(f"\nProcessing {split_name} ({total} rows) in {num_shards} shards...")

    for i in range(num_shards):
        shard_path = f"{shard_dir}/shard_{i}"

        # ── Skip already completed shards ────────────────────────────────────
        if os.path.exists(shard_path):
            print(f"  [SKIP] Shard {i+1}/{num_shards} already done.")
            continue

        start = i * SHARD_SIZE
        end   = min(start + SHARD_SIZE, total)
        print(f"  [RUN ] Shard {i+1}/{num_shards} — rows {start}:{end}")

        shard = dataset_split.select(range(start, end))

        mapped = shard.map(
            prepare_dataset,
            remove_columns=shard.column_names,
            num_proc=16,
            writer_batch_size=50,
            keep_in_memory=False,
        )

        # Filter corrupt audio
        before = len(mapped)
        mapped = mapped.filter(lambda x: x["is_valid"])
        mapped = mapped.remove_columns(["is_valid"])
        print(f"    Kept {len(mapped)}/{before} rows after filtering.")

        # ── Save shard immediately ────────────────────────────────────────────
        mapped.save_to_disk(shard_path)
        print(f"    ✓ Shard {i+1} saved.")

    # ── Concatenate all shards into final split ───────────────────────────────
    print(f"\n  Concatenating {num_shards} shards for {split_name}...")
    shards = [
        load_from_disk(f"{shard_dir}/shard_{i}")
        for i in range(num_shards)
    ]
    final = concatenate_datasets(shards)

    # ── Save final and clean up shards ───────────────────────────────────────
    final_path = f"{save_base}/{split_name}"
    final.save_to_disk(final_path)
    print(f"  ✓ {split_name} done — {len(final)} rows saved to {final_path}")

    # Optional: remove shard folder after successful merge
    import shutil
    shutil.rmtree(shard_dir)
    print(f"  Cleaned up shard folder.")

    return final
train_dataset = process_split_resumable(raw["train"], "train")
val_dataset   = process_split_resumable(raw["val"],   "val")
test_dataset  = process_split_resumable(raw["test"],  "test")

### 7a — Alternative Processor (Non-Resumable) & Label Verification

**What this does:** Two things:
1. Defines `process_split()` — a simpler version without sharding, useful for small datasets or re-running quickly. The commented-out lines at the top can load pre-saved processed data from disk instead of reprocessing.
2. Decodes and prints one processed label to verify tokenisation worked correctly.

**What to look for in the output:**
You should see special tokens surrounding the transcription text:
```
<|startoftranscript|> <|ur|> <|transcribe|> <|notimestamps|> بالکل <|endoftext|>
```
- `<|ur|>` (or `<|ps|>`, `<|pa|>`) → language tag ✅
- `<|transcribe|>` → task tag ✅
- `<|endoftext|>` at the end → clean termination ✅

> ⚠️ If the language tag shows `<|en|>` for a non-English clip, check the `LANG_MAPPING` in Step 3.


In [ ]:
"""   
Uncomment below if there are saved shards in a directory
"""

# print(dataset)
# dataset_train= load_from_disk("./processed/train")
# dataset_test= load_from_disk("./processed/test")
# dataset_val= load_from_disk("./processed/val")

def process_split(dataset_split, split_name):
    print(f"\nProcessing {split_name} ({len(dataset_split)} rows)...")
    mapped = dataset_split.map(
        prepare_dataset,
        remove_columns=dataset_split.column_names,
        num_proc=4,
        writer_batch_size=50,
        keep_in_memory=False,
    )
    # Drop rows where audio failed to load
    before = len(mapped)
    mapped = mapped.filter(lambda x: x["is_valid"])
    mapped = mapped.remove_columns(["is_valid"])
    print(f"  Kept {len(mapped)}/{before} rows after filtering corrupt files.")
    return mapped

# train_dataset = process_split(raw["train"], "train")
# val_dataset   = process_split(raw["val"],   "val")
# test_dataset  = process_split(raw["test"],  "test")

dataset_train = train_dataset
dataset_val = val_dataset
dataset_test = test_dataset
sample_idx = 0
decoded = processor.tokenizer.decode(
    dataset_train[sample_idx]["labels"],
    skip_special_tokens=False,
)
print(f"Sample label (with special tokens):\n{decoded}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Sample label (with special tokens):
<|startoftranscript|><|en|><|transcribe|><|notimestamps|>Let's say we say it.. Give it up.. Do you ?. Tell your uncle once.<|endoftext|>


## Step 8 — Define Label Length Filter

**What this does:** Defines a filter that will discard any training example whose transcription exceeds **448 tokens**.

> **Why 448?** Whisper's decoder can generate at most 448 tokens per clip. Training on longer examples would be wasted computation and can cause shape-mismatch errors.

> **Non-technical:** A bouncer at the door checking each guest's name fits on the nametag. If it doesn't fit, they can't come in.

> ℹ️ **This cell defines the filter function only. The next cell applies it.**


In [12]:
import numpy as np
def filter_long_labels_numpy(batch):
    lengths = np.array([
        len(l) if (l is not None and len(l) > 0) else 0   # handle empty list too
        for l in batch["labels"]
    ])
    return lengths <= MAX_LABEL_LENGTH

### 8a — Apply Filter & Remove Unnecessary Columns

**What this does:**
1. Applies the label-length filter to all three splits — removing overly long transcriptions
2. Drops all helper columns that are no longer needed, keeping only:
   - **`input_features`** — the Mel spectrogram (audio → numbers)
   - **`labels`** — the tokenised transcription (text → numbers)

After this cell, the datasets are fully prepared and clean — ready to go into the model.

> ⚠️ If the row count drops significantly here, your dataset may contain many very long transcriptions. Consider raising `MAX_LABEL_LENGTH` (Step 3) or reviewing your data.


In [13]:
train_dataset = train_dataset.filter(filter_long_labels_numpy,
    batched=True,
    batch_size=20000,
    # num_proc=4,
    writer_batch_size=500,
    desc="Filtering train"
)
val_dataset = val_dataset.filter(filter_long_labels_numpy,
    batched=True,
    batch_size=20000,
    # num_proc=1,
    writer_batch_size=500,
    desc="Filtering train"
)
test_dataset = test_dataset.filter(filter_long_labels_numpy,
    batched=True,
    batch_size=20000,
    # num_proc=4,
    writer_batch_size=500,
    desc="Filtering train"
)

print(f"After filtering → Train: {len(train_dataset)} | Val: {len(val_dataset)}")

train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c not in ["input_features", "labels"]]
)
val_dataset = val_dataset.remove_columns(
    [c for c in val_dataset.column_names if c not in ["input_features", "labels"]]
)
test_dataset = test_dataset.remove_columns(
    [c for c in test_dataset.column_names if c not in ["input_features", "labels"]]
)
print("Train rows:", len(train_dataset))
print("Val rows:", len(val_dataset))
print("Test rows:", len(test_dataset))

Filtering train: 100%|██████████| 118/118 [00:00<00:00, 27716.18 examples/s]

After filtering → Train: 936 | Val: 118
Train rows: 936
Val rows: 118
Test rows: 118


## Step 9 — Load Whisper Model & Attach LoRA Adapters

**What this does:**
1. **Loads Whisper Large v2** from Hugging Face into GPU memory in FP16 (half precision) to reduce memory usage
2. **Attaches LoRA adapters** to the model's attention layers — the only parts that will be updated during training

**LoRA configuration:**
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `r` | 128 | Rank of adapter matrices — higher = more expressive but more memory |
| `lora_alpha` | 256 | Scaling factor controlling adapter influence |
| Target layers | `q_proj, k_proj, v_proj, out_proj` | All four attention projections |
| `lora_dropout` | 0.05 | Light regularisation to prevent overfitting |

> **Non-technical:** The base Whisper model is an expert who already speaks 99 languages. The LoRA adapters are a small specialised notepad for taking notes on Urdu, Pashto, and Punjabi — the expert's existing knowledge stays untouched.

> ⏱️ First run downloads ~3 GB. Subsequent runs use the locally cached model.


In [14]:
print("\nLoading model...")

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda",          
)

model.config.use_cache = False
model.generation_config.language = None
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model.enable_input_require_grads()
print("\nApplying LoRA...")

lora_config = LoraConfig(
    r=128,                          
    lora_alpha=256,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,   
    )
model = get_peft_model(model, lora_config)



Loading model...


Loading weights: 100%|██████████| 1259/1259 [00:02<00:00, 426.01it/s]



Applying LoRA...


### 9a — Patch Forward Pass & Inspect Trainable Parameters

**What this does:**
1. Applies a small compatibility patch to the model's forward method so the PEFT/LoRA wrapper works correctly with Whisper's encoder-decoder architecture
2. Prints the ratio of trainable vs. frozen parameters

**What to expect in the output:**
```
trainable params: ~125M || all params: ~1.67B || trainable%: ~7.5%
```
Only ~7.5% of parameters are trainable — the LoRA adapters. The remaining ~92.5% (the frozen base model) will not change at all during training.

> **Non-technical:** We're only updating the sticky notes (LoRA adapters), not rewriting the whole textbook (base model).


In [15]:
def custom_forward(self, input_features, labels=None, **kwargs):
    kwargs.pop("input_ids", None)
    return self.base_model(input_features=input_features, labels=labels, **kwargs)

model.forward = custom_forward.__get__(model, type(model))

model.print_trainable_parameters()

trainable params: 125,829,120 || all params: 1,669,134,080 || trainable%: 7.5386


## Step 10 — Define Data Collator

**What this does:** Creates `DataCollatorSpeechSeq2SeqWithPadding` — a helper that assembles individual samples into batches during training. For each batch it:
- Pads audio feature grids so all clips in the batch have the same length
- Pads label sequences to match the longest transcription in the batch
- Masks padding positions with `-100` so the loss function ignores them (padding should not contribute to the score)
- Removes the beginning-of-sequence token from labels (Whisper generates it automatically)

> **Non-technical:** Audio clips vary in length, like sentences vary in word count. Before grouping them into a batch (a school class), this collator pads short clips with silence so every clip in the class has the same duration — like adding blank seats to make the row even.

> ℹ️ **This cell defines and instantiates the collator. No data is processed yet.**


In [16]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List, torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Input features - MUST be float16

        input_features = []
        for f in features:
            tensor = torch.tensor(f["input_features"])
            if tensor.shape[-1] > MAX_INPUT_LENGTH:
                tensor = tensor[..., :MAX_INPUT_LENGTH]
            input_features.append({"input_features": tensor})

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
            padding=True
        )
        

        # Labels
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt", padding=True
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


## Step 11 — Define Evaluation Metrics

**What this does:** Loads the two scoring functions used after each epoch to measure transcription quality.

| Metric | Stands for | Measures | Better when… |
|--------|-----------|----------|--------------|
| **WER** | Word Error Rate | % of words transcribed incorrectly | **Lower** — 0% = perfect |
| **BLEU** | Bilingual Evaluation Understudy | Overlap between prediction and reference | **Higher** — 100 = perfect |

> **Example WER:** If the model says *"He went to marked"* but the answer is *"He went to market"*, WER = 25% (1 word wrong out of 4).

> **Non-technical:** These are the model's exam scores. After every epoch the trainer checks both scores to decide if the model is improving. The trainer always saves the checkpoint with the **lowest WER**.


In [17]:
wer_metric = evaluate.load("wer")
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [s.strip() for s in pred_str]
    label_str = [s.strip() for s in label_str]

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    bleu = bleu_metric.compute(predictions=pred_str, references=[[r] for r in label_str])["score"]

    return {"wer": round(wer, 2), "bleu": round(bleu, 2)}

## Step 12 —  Evaluate Base Model on Test Set before Training



In [ ]:
eval_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results_merged_lora",
    per_device_eval_batch_size=16,
    predict_with_generate=True,     
    generation_max_length=448,
    )
# dataset_test=dataset_test.select(range(1,500))
trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    train_dataset=None,
    eval_dataset=dataset_test,  # Use your prepared validation/test dataset
    processing_class=processor.feature_extractor,
    data_collator=data_collator,          # <--- MAKE SURE THIS IS HERE
    compute_metrics=compute_metrics,
)

# 4. Run Evaluation
print("Running evaluation...")
results = trainer.evaluate()

print(f"Final WER: {results['eval_wer']}")
print(f"Final BLEU: {results['eval_bleu']}")

## Step 12 — Configure the Trainer

**What this does:** Builds the full training configuration and creates the `Seq2SeqTrainer` that manages the entire training loop.

**Key settings:**
| Setting | Value | What it means |
|---------|-------|--------------|
| Epochs | 5 | Model sees all training data 5 times |
| Effective batch size | 256 (32 × 8) | Samples per weight update |
| Learning rate | 1e-4 with cosine decay | Starts at 0.0001, decreases smoothly |
| Warmup | 10% of steps | LR ramps up slowly at start to prevent instability |
| Best model metric | WER ↓ | Always keeps the checkpoint with lowest WER |
| Checkpoint limit | 2 | Only 2 most recent best checkpoints kept |
| Precision | FP16 | Half-precision — halves GPU memory with minimal quality loss |
| Logging | TensorBoard | Training metrics written for live visualisation |

> **Non-technical:** This is setting the oven temperature, rack position, and timer before pressing Start.

> 💡 To monitor training live, open a terminal and run:
> ```bash
> tensorboard --logdir whisper_finetuning/
> ```


In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    fp16=True,
    report_to=["tensorboard"],
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_pin_memory=False,
    dataloader_num_workers=4,
    optim="adamw_torch_fused",
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Step 13 — 🚀 Start Training

**What this does:** Starts the training loop. Each epoch:
1. Feeds batches of (audio features, labels) through the model
2. Computes the prediction error (loss)
3. Updates the LoRA adapter weights to reduce that error
4. Evaluates WER and BLEU on the validation set
5. Saves a checkpoint if the model improved

**What you will see during training:**
- A progress bar showing steps within the current epoch
- After each epoch: a table with `train_loss`, `eval_loss`, `wer`, `bleu`
- `Saving model checkpoint to …` when a checkpoint is written

> ⚠️ **Training is resumable.** If interrupted, just re-run this cell — it will automatically continue from the last saved checkpoint.

> ⏱️ Expected duration: **several hours** depending on GPU speed and dataset size.

> **Non-technical:** The model is now in school, studying. Each epoch is one school day. At the end of each day, the teacher gives a test (WER/BLEU on validation data) and saves the student's best work.


In [ ]:
print("\nStarting training...")

last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    from transformers.trainer_utils import get_last_checkpoint
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

trainer.train(resume_from_checkpoint=last_checkpoint)


## Step 14 — 💾 Save the Trained Model

**What this does:** Saves the model in two formats:

| Format | Location | Size | Use case |
|--------|----------|------|---------|
| **LoRA Adapter** | `OUTPUT_DIR/adapter/` | ~50–200 MB | Lightweight; apply on top of any Whisper Large v2 copy |
| **Merged Model** | `OUTPUT_DIR/merged/` | ~3 GB | Self-contained; ready for deployment without the PEFT library |

> **Non-technical:** The adapter is like saving just your sticky notes from the textbook. The merged model is like printing a brand-new edition of the textbook with your notes already printed inside.

> ✅ After this cell completes, your trained model is ready to use for speech recognition.


In [ ]:
print("\nSaving model...")

adapter_path = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

# Merge LoRA weights
merged_model = model.merge_and_unload()
merged_path = f"{OUTPUT_DIR}/merged"
merged_model.save_pretrained(merged_path)
processor.save_pretrained(merged_path)

print(f"Training completed! Adapter saved at: {adapter_path}")
print(f"Merged model saved at: {merged_path}")

## Evaluate the Model On Test Set After Training 

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(merged_path).to("cuda")
processor = WhisperProcessor.from_pretrained(merged_path)

eval_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results_merged_lora",
    per_device_eval_batch_size=16,
    predict_with_generate=True,     
    generation_max_length=448,
    )
# dataset_test=dataset_test.select(range(1,500))
trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    train_dataset=None,
    eval_dataset=dataset_test,  # Use your prepared validation/test dataset
    processing_class=processor.feature_extractor,
    data_collator=data_collator,          # <--- MAKE SURE THIS IS HERE
    compute_metrics=compute_metrics,
)

# 4. Run Evaluation
print("Running evaluation...")
results = trainer.evaluate()

print(f"Final WER: {results['eval_wer']}")
print(f"Final BLEU: {results['eval_bleu']}")

## Convert to CTranslate2 Format
  Convertion of Model to CTranslate2 is required for Whisperx Library. If you are not using whisperx library, skip this step. 

!pip install ctranslate2

In [ ]:
ct2-transformers-converter  --model OUTPUT_DIR/merged/  --output_dir OUTPUT_DIR/merged_C2T  --force --quantization float16 --copy_files tokenizer.json preprocessor_config.json 
